In [2]:
# Load the autoreload extension
%load_ext autoreload

# Set autoreload mode
%autoreload 2

# Scraping Analysis & Discussion

In [3]:
from coralnet_scraper import CoralNetDownloader
import getpass
import boto3
import numpy as np
import pandas as pd
import tqdm

In [4]:
def check_s3_prefix_exists(bucket_name, s3_prefix, source_id, specific_file = "annotations.csv"):
    s3 = boto3.client("s3")
    prefix = f"{s3_prefix}/s{source_id}/{specific_file}"

    response = s3.list_objects_v2(Bucket=bucket_name, Prefix=prefix, MaxKeys=1)

    if "Contents" in response:
        # print(f"Prefix exists: {prefix}")
        return True
    else:
        # print(f"Prefix does not exist: {prefix}")
        return False

# Image List Issue Fixes 
- Here we investigate the cases where the image_list.csv doesn't contain all of the images due to a bug in the scraping code re-writing the dataframe. There might be other discrepancies (such as fewer images that have been downloaded and such that need to be investigated in the future). 

## Get sources with the image_list bug

In [5]:
df_counts = pd.read_csv("../nbs/EDA/dataframes/coralnet_source_counts.csv")
df_counts

,source_id,image_list_count,annotations_count,images_folder_count
0,23,750,750,750
1,57,1681,1681,1681
2,69,100,100,100
3,70,300,300,300
4,82,1,0,1
...,...,...,...,...
1956,8277,0,0,0
1957,8278,0,0,0
1958,8284,0,0,0
1959,8288,0,0,0


In [6]:
df_counts_problematic = df_counts[df_counts["image_list_count"]!=df_counts["images_folder_count"]]
df_counts_problematic = df_counts_problematic[df_counts_problematic["image_list_count"]<df_counts_problematic["images_folder_count"]]

In [66]:
print(f"There are {df_counts_problematic.shape[0]} problematic sources, containing a total of {df_counts_problematic['images_folder_count'].sum()-df_counts_problematic['image_list_count'].sum()} missing images.")

There are 32 problematic sources, containing a total of 505975 missing images.


In [65]:
df_counts_problematic.sort_values(by="images_folder_count").head(10)

,source_id,image_list_count,annotations_count,images_folder_count,image_list_count_upd
908,5027,40,40,41,40
217,2616,1904,4904,4904,4904
216,2615,943,4943,4943,4943
150,2112,89,6089,6089,6089
318,3015,1081,7081,7081,7081
299,2947,1143,7143,7142,7143
459,3466,1202,7202,7202,7202
503,3551,1202,7202,7202,7202
510,3567,1419,7419,7419,7419
251,2795,1779,7779,7779,7779


## Redownload image-list

In [ ]:
# username = input("CoralNet username: ")
# password = getpass.getpass("CoralNet password: ")

# source_ids = df_counts_problematic["source_id"].values

# bucket_name = "dev-datamermaid-sm-sources"
# prefix = "coralnet-public-images"
# s3 = boto3.client("s3")

# downloader = CoralNetDownloader(username=username, password=password)
# for source_id in tqdm.tqdm(["295"], desc = "sources"):
#     s3_key_output = f"{prefix}/imagelist/s{source_id}_image_list.csv"
#     response = s3.list_objects_v2(Bucket=bucket_name, Prefix=s3_key_output, MaxKeys=1)
#     if "Contents" in response:
#         continue 
#     df_images, success = downloader.get_images(source_id=source_id)
#     if success==True and len(df_images)>0: 
#         s3_key_output = f"{prefix}/imagelist/s{source_id}_image_list.csv"
#         s3.put_object(
#             Bucket=bucket_name,
#             Key=s3_key_output,
#             Body=df_images.to_csv(index=False)
#         )

## Look into the descrepancies for an individual source

In [10]:
source_ids = df_counts_problematic["source_id"].values

bucket_name = "dev-datamermaid-sm-sources"
prefix = "coralnet-public-images"
s3 = boto3.client("s3")

In [43]:
source_id = 2616
s3_key_image = f"{prefix}/s{source_id}/image_list.csv"
obj_il = s3.get_object(Bucket=bucket_name, Key=s3_key_image)
image_list_df = pd.read_csv(obj_il["Body"])

In [44]:
# Assuming we already have the rescraped images
s3_key_image = f"{prefix}/imagelist/s{source_id}_image_list.csv"
obj_il = s3.get_object(Bucket=bucket_name, Key=s3_key_image)
df_images = pd.read_csv(obj_il["Body"])

In [17]:
s3_key_annotations = f"{prefix}/s{source_id}/annotations.csv"
obj_ann = s3.get_object(Bucket=bucket_name, Key=s3_key_annotations)
annotations_df = pd.read_csv(obj_ann["Body"])

/tmp/ipykernel_73040/1466276775.py:3: DtypeWarning: Columns (10) have mixed types. Specify dtype option on import or set low_memory=False.
  annotations_df = pd.read_csv(obj_ann["Body"])


In [18]:
images_prefix = f"{prefix}/s{source_id}/images/"

paginator = s3.get_paginator("list_objects_v2")
file_count = 0

image_list = []
for page in paginator.paginate(Bucket=bucket_name, Prefix=images_prefix):
    for obj in page.get("Contents", []):
        key = obj["Key"]
        if key != images_prefix and not key.endswith("/"):
            image_list.append(key)
            file_count += 1

In [19]:
annotations_df.head(3)

,Name,Date,Season,Shelf-position,Reef,Exposure,Zone,Height (cm),Latitude,Longitude,...,Strobes,Framing gear used,White balance card,Comments,Row,Column,Label code,Label ID,Annotator,Date annotated
0,2019_Fall_Fahal_Lee_Crest_T1.IMG_1670.JPG,2021-10-31,Fall,Midshelf,Al Fahal,Sheltered,Crest,50,NaN,NaN,...,No,No,No,NaN,547,999,Pavement,2513,robot,2022-01-18 14:34:19+00:00
1,2019_Fall_Fahal_Lee_Crest_T1.IMG_1670.JPG,2021-10-31,Fall,Midshelf,Al Fahal,Sheltered,Crest,50,NaN,NaN,...,No,No,No,NaN,290,576,Pavement,2513,robot,2022-01-18 14:34:19+00:00
2,2019_Fall_Fahal_Lee_Crest_T1.IMG_1670.JPG,2021-10-31,Fall,Midshelf,Al Fahal,Sheltered,Crest,50,NaN,NaN,...,No,No,No,NaN,641,733,Shadow,222,robot,2022-01-18 14:34:19+00:00


In [20]:
image_list_df.head(3)

,Name,Image Page,Image URL
0,2019_Summ_Shark_Lee_Crest_T3IMG_0777.JPG - Con...,/image/2064680/view/,https://coralnet-production.s3.amazonaws.com/m...
1,2019_Summ_Shark_Lee_Crest_T3IMG_0778.JPG - Con...,/image/2064684/view/,https://coralnet-production.s3.us-west-2.amazo...
2,2019_Summ_Shark_Lee_Crest_T3IMG_0779.JPG - Con...,/image/2064688/view/,https://coralnet-production.s3.amazonaws.com/m...


In [23]:
df_images.head()

,Name,Image Page
0,2019_Fall_Fahal_Lee_Crest_T1.IMG_1670.JPG - Un...,/image/2274993/view/
1,2019_Fall_Fahal_Lee_Crest_T1.IMG_1671.JPG - Un...,/image/2274996/view/
2,2019_Fall_Fahal_Lee_Crest_T1.IMG_1672.JPG - Un...,/image/2275000/view/
3,2019_Fall_Fahal_Lee_Crest_T1.IMG_1673.JPG - Un...,/image/2275003/view/
4,2019_Fall_Fahal_Lee_Crest_T1.IMG_1674.JPG - Un...,/image/2275006/view/


In [30]:
# Extract image names from both dataframes
df_images_names = set(df_images['Name'].str.extract(r'([^-]+)')[0])
image_list_names = set(image_list_df['Name'].str.extract(r'([^-]+)')[0])

# Find intersection
intersection = df_images_names & image_list_names

print(f"Intersection count: {len(intersection)}")
print(f"Images in df_images only: {len(df_images_names - image_list_names)}")
print(f"Images in image_list_df only: {len(image_list_names - df_images_names)}")

Intersection count: 1904
Images in df_images only: 3000
Images in image_list_df only: 0


In [31]:
annotations_df["Name"].nunique(), len(image_list_df), len(image_list), len(df_images)

(4904, 1904, 4904, 4904)

## Check counts of new sources

In [46]:
images_prefix = f"{prefix}/imagelist/"

paginator = s3.get_paginator("list_objects_v2")
file_count = 0

image_list = []
for page in paginator.paginate(Bucket=bucket_name, Prefix=images_prefix):
    for obj in page.get("Contents", []):
        key = obj["Key"]
        if key != images_prefix and not key.endswith("/"):
            image_list.append(key)
            file_count += 1


In [51]:
image_list[:3]

['coralnet-public-images/imagelist/s1580_image_list.csv',
 'coralnet-public-images/imagelist/s2112_image_list.csv',
 'coralnet-public-images/imagelist/s2615_image_list.csv']

In [49]:
image_list_count_upd = []
for source_id in df_counts_problematic["source_id"]:
    try: 
        s3_key_output = f"{prefix}/imagelist/s{source_id}_image_list.csv"
        obj_il = s3.get_object(Bucket=bucket_name, Key=s3_key_output)
        image_list_df = pd.read_csv(obj_il["Body"])
        image_list_count_upd.append(image_list_df.shape[0])
    except Exception as e:
        print(f"Error occurred while processing source {source_id}: {e}")
        image_list_count_upd.append(np.nan)
        continue

In [50]:
df_counts_problematic["image_list_count_upd"] = image_list_count_upd

In [52]:
df_counts_problematic

,source_id,image_list_count,annotations_count,images_folder_count,image_list_count_upd
12,295,1064,79064,79062,79064
16,372,98,66098,66098,66098
17,373,2485,29485,29485,29485
24,554,1395,22395,22393,22395
84,1580,1421,16421,16421,16421
150,2112,89,6089,6089,6089
216,2615,943,4943,4943,4943
217,2616,1904,4904,4904,4904
251,2795,1779,7779,7779,7779
292,2897,2512,29512,29512,37454


# Analyze Confirmed/Unconfirmed Cases
- Within the imagelist each image has one of the following suffixes: " - Confirmed", " - Unconfirmed", " - Unclassified". However, in the annotations dataframe, each image only has the image ID (no such suffix). As I previously did not know this, I added a solution to remove the " - Confirmed" suffix before the imagelist dataframe is merged with the annotations dataframe, which consequently led to only those images being merged (all of the other image paths were specified as NaN which is why a large portion of images were missing).  

In [55]:
source_ids = df_counts_problematic["source_id"].values

bucket_name = "dev-datamermaid-sm-sources"
prefix = "coralnet-public-images"
s3 = boto3.client("s3")

In [56]:
source_id = 2616
s3_key_image = f"{prefix}/s{source_id}/image_list.csv"
obj_il = s3.get_object(Bucket=bucket_name, Key=s3_key_image)
image_list_df = pd.read_csv(obj_il["Body"])

In [57]:
# Assuming we already have the rescraped images
s3_key_image = f"{prefix}/imagelist/s{source_id}_image_list.csv"
obj_il = s3.get_object(Bucket=bucket_name, Key=s3_key_image)
df_images = pd.read_csv(obj_il["Body"])

In [58]:
s3_key_annotations = f"{prefix}/s{source_id}/annotations.csv"
obj_ann = s3.get_object(Bucket=bucket_name, Key=s3_key_annotations)
annotations_df = pd.read_csv(obj_ann["Body"])

/tmp/ipykernel_73040/1466276775.py:3: DtypeWarning: Columns (10) have mixed types. Specify dtype option on import or set low_memory=False.
  annotations_df = pd.read_csv(obj_ann["Body"])


In [59]:
images_prefix = f"{prefix}/s{source_id}/images/"

paginator = s3.get_paginator("list_objects_v2")
file_count = 0

image_list = []
for page in paginator.paginate(Bucket=bucket_name, Prefix=images_prefix):
    for obj in page.get("Contents", []):
        key = obj["Key"]
        if key != images_prefix and not key.endswith("/"):
            image_list.append(key)
            file_count += 1

In [60]:
image_list_df["Status"] = None
confirmed_mask = image_list_df["Name"].apply(lambda x: "Confirmed" in x)
Unconfirmed_mask = image_list_df["Name"].apply(lambda x: "Unconfirmed" in x)
Unclassified_mask = image_list_df["Name"].apply(lambda x: "Unclassified" in x)
image_list_df.loc[confirmed_mask, "Status"] = "Confirmed"
image_list_df.loc[Unconfirmed_mask, "Status"] = "Unconfirmed"
image_list_df.loc[Unclassified_mask, "Status"] = "Unclassified"

df_images["Status"] = None
confirmed_mask = df_images["Name"].apply(lambda x: "Confirmed" in x)
Unconfirmed_mask = df_images["Name"].apply(lambda x: "Unconfirmed" in x)
Unclassified_mask = df_images["Name"].apply(lambda x: "Unclassified" in x)
df_images.loc[confirmed_mask, "Status"] = "Confirmed"
df_images.loc[Unconfirmed_mask, "Status"] = "Unconfirmed"
df_images.loc[Unclassified_mask, "Status"] = "Unclassified"

In [61]:
image_list_df["Status"].value_counts()

Status
Unconfirmed    1301
Confirmed       603
Name: count, dtype: int64

In [62]:
df_images["Status"].value_counts()

Status
Unconfirmed    2497
Confirmed      2407
Name: count, dtype: int64

In [33]:
df_annotations = annotations_df.copy()
df_images_copy = df_images.copy()
df_images["Name"] = df_images["Name"].apply(
    lambda x: x.replace(" - Confirmed", "")
)
df_images["image_id"] = df_images["Image Page"].apply(
    lambda x: x.replace("/image/", "").replace("/view/", "")
)
df_annotations_copy = pd.merge(
    df_annotations,
    df_images,
    left_on="Name",
    right_on="Name",
    how="left",
    suffixes=("", "_y"),
)

In [37]:
df_images.head()

,Name,Image Page,Status,image_id
0,2019_Fall_Fahal_Lee_Crest_T1.IMG_1670.JPG - Un...,/image/2274993/view/,Unconfirmed,2274993
1,2019_Fall_Fahal_Lee_Crest_T1.IMG_1671.JPG - Un...,/image/2274996/view/,Unconfirmed,2274996
2,2019_Fall_Fahal_Lee_Crest_T1.IMG_1672.JPG - Un...,/image/2275000/view/,Unconfirmed,2275000
3,2019_Fall_Fahal_Lee_Crest_T1.IMG_1673.JPG - Un...,/image/2275003/view/,Unconfirmed,2275003
4,2019_Fall_Fahal_Lee_Crest_T1.IMG_1674.JPG - Un...,/image/2275006/view/,Unconfirmed,2275006


In [36]:
df_annotations_copy.head()

,Name,Date,Season,Shelf-position,Reef,Exposure,Zone,Height (cm),Latitude,Longitude,...,Comments,Row,Column,Label code,Label ID,Annotator,Date annotated,Image Page,Status,image_id
0,2019_Fall_Fahal_Lee_Crest_T1.IMG_1670.JPG,2021-10-31,Fall,Midshelf,Al Fahal,Sheltered,Crest,50,NaN,NaN,...,NaN,547,999,Pavement,2513,robot,2022-01-18 14:34:19+00:00,NaN,NaN,NaN
1,2019_Fall_Fahal_Lee_Crest_T1.IMG_1670.JPG,2021-10-31,Fall,Midshelf,Al Fahal,Sheltered,Crest,50,NaN,NaN,...,NaN,290,576,Pavement,2513,robot,2022-01-18 14:34:19+00:00,NaN,NaN,NaN
2,2019_Fall_Fahal_Lee_Crest_T1.IMG_1670.JPG,2021-10-31,Fall,Midshelf,Al Fahal,Sheltered,Crest,50,NaN,NaN,...,NaN,641,733,Shadow,222,robot,2022-01-18 14:34:19+00:00,NaN,NaN,NaN
3,2019_Fall_Fahal_Lee_Crest_T1.IMG_1670.JPG,2021-10-31,Fall,Midshelf,Al Fahal,Sheltered,Crest,50,NaN,NaN,...,NaN,352,1751,Pavement,2513,robot,2022-01-18 14:34:19+00:00,NaN,NaN,NaN
4,2019_Fall_Fahal_Lee_Crest_T1.IMG_1670.JPG,2021-10-31,Fall,Midshelf,Al Fahal,Sheltered,Crest,50,NaN,NaN,...,NaN,310,2642,Pavement,2513,robot,2022-01-18 14:34:19+00:00,NaN,NaN,NaN


In [40]:
image_count_dict = {}

for source_id in tqdm.tqdm(df_counts["source_id"].values):
    image_list_flag = check_s3_prefix_exists(
        bucket_name=bucket_name, s3_prefix=prefix, source_id=source_id, specific_file="image_list.csv"
    )
    if not image_list_flag:
        # print(f"Source {source_id} does not have an image_list.csv file. Skipping.")
        continue
    s3_key_image = f"{prefix}/s{source_id}/image_list.csv"
    obj_il = s3.get_object(Bucket=bucket_name, Key=s3_key_image)
    image_list_df = pd.read_csv(obj_il["Body"])

    image_list_df["Status"] = "Other"
    confirmed_mask = image_list_df["Name"].apply(lambda x: "Confirmed" in x)
    Unconfirmed_mask = image_list_df["Name"].apply(lambda x: "Unconfirmed" in x or "Unclassified" in x)
    image_list_df.loc[confirmed_mask, "Status"] = "Confirmed"
    image_list_df.loc[Unconfirmed_mask, "Status"] = "Unconfirmed"
    if "Other" in image_list_df["Status"].unique():
        print(f"Source {source_id} has {image_list_df[image_list_df['Status']=='Other'].shape[0]} images with 'Other' status.")
        if image_list_df[image_list_df["Status"] == "Other"].shape[0] > 100:
            break
    for status in image_list_df["Status"].unique():
        count = image_list_df[image_list_df["Status"] == status].shape[0]
        if status not in image_count_dict:
            image_count_dict[status] = 0
        image_count_dict[status] += count


  0%|          | 1/1961 [00:00<06:31,  5.00it/s]

100%|██████████| 1961/1961 [04:04<00:00,  8.01it/s]


In [41]:
image_count_dict

{'Confirmed': 486155, 'Unconfirmed': 180677}

In [42]:
image_list_df

,Name,Image Page,Image URL,Status
0,Aleopora tizardi 平滑穴孔珊瑚.png - Unclassified,/image/5807430/view/,https://coralnet-production.s3.us-west-2.amazo...,Unconfirmed
1,Galaxea astreata 稀杯盔形珊瑚.jpg - Unclassified,/image/5807433/view/,https://coralnet-production.s3.amazonaws.com/m...,Unconfirmed
2,Galaxea fascicularis 丛生盔形珊瑚.jpg - Unclassified,/image/5807431/view/,https://coralnet-production.s3.amazonaws.com/m...,Unconfirmed
3,Galaxea fascicularis 丛生盔形珊瑚.png - Unclassified,/image/5807432/view/,https://coralnet-production.s3.amazonaws.com/m...,Unconfirmed
4,Goniopora columna 柱状角孔珊瑚.jpg - Unclassified,/image/5807429/view/,https://coralnet-production.s3.us-west-2.amazo...,Unconfirmed
5,Goniopora minor 小角孔珊瑚.png - Unclassified,/image/5807428/view/,https://coralnet-production.s3.us-west-2.amazo...,Unconfirmed
6,Porites australiensis 澳洲滨珊瑚.jpg - Unclassified,/image/5807416/view/,https://coralnet-production.s3.us-west-2.amazo...,Unconfirmed
7,Porites cylindrical 细柱滨珊瑚.png - Unclassified,/image/5807422/view/,https://coralnet-production.s3.amazonaws.com/m...,Unconfirmed
8,Porites cylindrica 细柱滨珊瑚.jpg - Unclassified,/image/5807412/view/,https://coralnet-production.s3.amazonaws.com/m...,Unconfirmed
9,Porites lichen 地衣滨珊瑚.png - Unclassified,/image/5807421/view/,https://coralnet-production.s3.us-west-2.amazo...,Unconfirmed


# Confirmed Reanalysis

In [71]:
image_count_dict = {}

for _, row in tqdm.tqdm(df_counts.iterrows()):
    source_id = row["source_id"]
    problematic = row["image_list_count"]<row["images_folder_count"]
    if problematic:
        s3_key_output = f"{prefix}/imagelist/s{source_id}_image_list.csv"
        obj_il = s3.get_object(Bucket=bucket_name, Key=s3_key_output)
        image_list_df = pd.read_csv(obj_il["Body"])
    else:
        image_list_flag = check_s3_prefix_exists(
            bucket_name=bucket_name, s3_prefix=prefix, source_id=source_id, specific_file="image_list.csv"
        )
        if not image_list_flag:
            # print(f"Source {source_id} does not have an image_list.csv file. Skipping.")
            continue
        s3_key_image = f"{prefix}/s{source_id}/image_list.csv"
        obj_il = s3.get_object(Bucket=bucket_name, Key=s3_key_image)
        image_list_df = pd.read_csv(obj_il["Body"])

    image_list_df["Status"] = "Other"
    confirmed_mask = image_list_df["Name"].apply(lambda x: "Confirmed" in x)
    Unconfirmed_mask = image_list_df["Name"].apply(lambda x: "Unconfirmed" in x)
    Unclassified_mask = image_list_df["Name"].apply(lambda x: "Unclassified" in x)
    image_list_df.loc[confirmed_mask, "Status"] = "Confirmed"
    image_list_df.loc[Unconfirmed_mask, "Status"] = "Unconfirmed"
    image_list_df.loc[Unclassified_mask, "Status"] = "Unclassified"
    if "Other" in image_list_df["Status"].unique():
        print(f"Source {source_id} has {image_list_df[image_list_df['Status']=='Other'].shape[0]} images with 'Other' status.")
        if image_list_df[image_list_df["Status"] == "Other"].shape[0] > 100:
            break
    for status in image_list_df["Status"].unique():
        count = image_list_df[image_list_df["Status"] == status].shape[0]
        if status not in image_count_dict:
            image_count_dict[status] = 0
        image_count_dict[status] += count

1it [00:00,  3.18it/s]

1961it [03:44,  8.73it/s]


In [72]:
image_count_dict

{'Confirmed': 795860, 'Unconfirmed': 359271, 'Unclassified': 29470}